In [ ]:
import numpyro
import numpyro.distributions as dist
import numpyro.distributions.constraints as constraints
from numpyro.infer.reparam import TransformReparam

In [ ]:
import jax
import jax.numpy as jnp
from jax import random

In [ ]:
import numpy as np
import math
from itertools import product, accumulate
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns
import umap
from functools import reduce
from tqdm import trange

In [ ]:
def hdmm_model(data, struct_upbd, vocab_size):
    
    K = struct_upbd["G0"]
    S = struct_upbd["G1"]
    C = struct_upbd["G2"]

    struct_params = {}
    gamma = numpyro.param("model_gamma", jnp.ones(K), constraint=constraints.positive)
    struct_params["model_gamma"] = gamma

    struct_dists = {}
    struct_dists["model_G0"] = numpyro.sample("G0", dist.Dirichlet(gamma))

    for s_idx in range(S):
        alpha = numpyro.param(f"model_alpha_S_{s_idx}", jnp.ones(K), constraint=constraints.positive)
        base_dist = struct_dists["model_G0"]

        struct_params[f"model_alpha_S_{s_idx}"] = alpha
        struct_dists[f"model_G1_S_{s_idx}"] = numpyro.sample(f"G1_S_{s_idx}", dist.Dirichlet(alpha*base_dist))

        for c_idx in range(C):
            alpha = numpyro.param(f"model_alpha_S_{s_idx}_C_{c_idx}", jnp.ones(K), constraint=constraints.positive)
            base_dist = struct_dists[f"model_G1_S_{s_idx}"]

            struct_params[f"model_alpha_S_{s_idx}_C_{c_idx}"] = alpha
            struct_dists[f"model_G2_S_{s_idx}_C_{c_idx}"] = numpyro.sample(f"G2_S_{s_idx}_C_{c_idx}", dist.Dirichlet(alpha*base_dist))
            struct_params[f"model_alpha_S_{s_idx}_C_{c_idx}_G"] = numpyro.param(f"model_alpha_S_{s_idx}_C_{c_idx}_G", jnp.ones(K), constraint=constraints.positive)

    cluster_params = {}
    cluster_dists = {}
    eta = numpyro.param("model_eta_G0", jnp.ones(S), constraint=constraints.positive)
    cluster_params["model_eta_G0"] = eta
    cluster_dists["model_G0"] = numpyro.sample("cluster_G0", dist.Dirichlet(eta))
    for s_idx in range(S):
        eta_1 = numpyro.param(f"model_eta_G1_S_{s_idx}", jnp.ones(C), constraint=constraints.positive)
        cluster_params[f"model_eta_G1_S_{s_idx}"] = eta_1
        cluster_dists[f"model_G1_S_{s_idx}"] = numpyro.sample(f"cluster_G1_S_{s_idx}", dist.Dirichlet(eta_1))

    beta = numpyro.param("model_beta", jnp.ones(vocab_size), constraint=constraints.positive)
    struct_params["model_beta"] = beta
    nig_alpha = numpyro.param("model_nig_alpha", jnp.array(2.), constraint=constraints.positive)
    struct_params["model_nig_alpha"] = nig_alpha
    nig_beta = numpyro.param("model_nig_beta", jnp.array(1.), constraint=constraints.positive)
    struct_params["model_nig_beta"] = nig_beta
    nig_mu = numpyro.param("model_nig_mu", jnp.array(0.), constraint=constraints.positive)
    struct_params["model_nig_mu"] = nig_mu
    nig_lam = numpyro.param("model_nig_lam", jnp.array(1.), constraint=constraints.positive)
    struct_params["model_nig_lam"] = nig_lam

    struct_dists["model_Gen"] = []
    struct_dists["model_Reg"] = []
    for k_idx in range(K):
        multi = numpyro.sample(f"Gen_{k_idx}", dist.Dirichlet(beta))
        struct_dists["model_Gen"].append(multi)
        sig2 = numpyro.sample("Reg_sigma2", dist.InverseGamma(nig_alpha, nig_beta))
        mu = numpyro.sample("Reg_mu", dist.Normal(nig_mu, jnp.sqrt(sig2 / nig_lam)))
        struct_dists["model_Reg"].append((mu, sig2))

    feature = data[0]
    label = data[1]
    assert feature.shape[0] == label.shape[0]
    for i in range(feature.shape[0]):
        cat = []
        z_s = numpyro.sample(f"z_S_{i}", dist.Categorical(logits=cluster_dists["model_G0"]))
        cat.append(z_s)
        z_c = numpyro.sample(f"z_C_{i}", dist.Categorical(logits=cluster_dists[f"model_G1_S_{z_s}"]))
        cat.append(z_c)
        g = numpyro.sample(f"z_G_{i}", dist.Dirichlet(struct_dists[f"model_G2_S_{z_s}_C_{z_c}"]))
        z_reg = numpyro.sample(f"z_Reg_{i}", dist.Categorical(logits=g))
        cat.append(z_reg)
        reg = numpyro.sample(f"obs_Reg_{i}", dist.Normal(struct_dists["model_Reg"][z_reg][0], jnp.sqrt(struct_dists["model_Reg"][z_reg][1])), obs=label[i])

        word_cats = []
        for m in feature.shape[1]:
            z_gen = numpyro.sample(f"z_Gen_{i}_{m}", dist.Categorical(logits=g))
            ob = numpyro.sample(f"obs_Gen_{i}_{m}", dist.Multinomial(1, struct_dists["model_Gen"][z_gen]), obs=feature[i, m, :])
            word_cats.append(z_gen)
        cat.append(word_cats)
    return {
        "struct_assumptions": struct_upbd,
        "latent_assignment": cat,
        "struct_params": struct_params,
        "struct_dists": struct_dists,
        "cluster_params": cluster_params,
        "cluster_dists": cluster_dists,
        "observed_variables": {
            "labels": label,
            "features": feature
        },
    }

In [ ]:
import jax
import jax.numpy as jnp
from jax import random
from jax.scipy.special import gammaln
from functools import partial

# ---- small helpers -----------------------------------------------------------

def _safe_log(x, eps=1e-12):
    return jnp.log(jnp.clip(x, eps, None))

def _log_normal_pdf(y, mu, var):
    return -0.5 * (_safe_log(2*jnp.pi*var) + (y - mu) ** 2 / var)

def _dirichlet_log_pdf(x, alpha):
    # log Dir(x | alpha) up to normalization; uses standard formula with gammaln
    alpha0 = jnp.sum(alpha)
    return jnp.sum((alpha - 1.0) * _safe_log(x)) + gammaln(alpha0) - jnp.sum(gammaln(alpha))

def _sample_categorical(key, log_probs):
    # log_probs: shape (..., K)
    return random.categorical(key, log_probs, axis=-1)

def _one_hot(idx, K):
    return jnp.eye(K, dtype=jnp.int32)[idx]

# Count words assigned to topic k across all docs/tokens (features are one-hot)
def _word_counts_for_topic(z_gen, features, K, V):
    # z_gen: (N, M) int in [0,K)
    # features: (N, M, V) one-hot
    counts = jnp.zeros((K, V))
    for k in range(K):
        mask = (z_gen == k)[..., None]  # (N, M, 1)
        counts = counts.at[k].set(jnp.sum(features * mask, axis=(0,1)))
    return counts  # (K, V)

def _posterior_nig(y_k, mu0, lam0, a0, b0):
    n = y_k.shape[0]
    if n == 0:
        return mu0, lam0, a0, b0
    ybar = jnp.mean(y_k)
    sse = jnp.sum((y_k - ybar)**2)
    lam_n = lam0 + n
    mu_n  = (lam0*mu0 + n*ybar) / lam_n
    a_n   = a0 + 0.5*n
    b_n   = b0 + 0.5*sse + 0.5*(lam0*n/lam_n) * (ybar - mu0)**2
    return mu_n, lam_n, a_n, b_n

# ---- main Gibbs --------------------------------------------------------------

def gibbs_hdmm(
    rng_key,
    model_ret,
    num_iters=200,
    burnin=100,
    thin=1,
    init_state=None,
    record_last_only=True,
):
    """
    Gibbs sampler for your hdmm_model using the returned 'model_ret' as priors.

    Parameters
    ----------
    rng_key : jax.random.PRNGKey
    model_ret : dict
        Output of hdmm_model(...).
    num_iters : int
        Total Gibbs sweeps.
    burnin : int
        Number of initial sweeps to discard (no effect if record_last_only=True).
    thin : int
        Keep 1 sample every `thin` sweeps (no effect if record_last_only=True).
    init_state : dict or None
        Optional initial latent state: keys 'z_S', 'z_C', 'z_Reg', 'z_Gen', 'g'.
    record_last_only : bool
        If True, only the final state/params are returned; otherwise, a trace is returned.

    Returns
    -------
    out : dict
        Latents and parameters (last or trace).
    """
    # ----- unpack priors / fixed structure -----
    struct = model_ret
    upbd = struct["struct_assumptions"]
    K = upbd["G0"]  # topics / components at bottom level
    S = upbd["G1"]  # first-level groups
    C = upbd["G2"]  # second-level groups

    features = struct["observed_variables"]["features"]  # (N, M, V), one-hot
    labels   = struct["observed_variables"]["labels"]    # (N,)
    N, M, V  = features.shape

    # cluster priors over hierarchy (these are probabilities from the model return)
    pi_S = jnp.asarray(struct["cluster_dists"]["model_G0"])               # (S,)
    pi_C = {s: jnp.asarray(struct["cluster_dists"][f"model_G1_S_{s}"])    # (C,)
           for s in range(S)}

    # structural base Dirichlets for per-item K-mixture
    G2 = {(s, c): jnp.asarray(struct["struct_dists"][f"model_G2_S_{s}_C_{c}"])  # (K,)
          for s in range(S) for c in range(C)}

    # priors for phi_k (words) and for (mu_k, sigma2_k) (regression)
    beta_dir = jnp.asarray(struct["struct_params"]["model_beta"])         # (V,)
    a0 = jnp.asarray(struct["struct_params"]["model_nig_alpha"])          # scalar > 0
    b0 = jnp.asarray(struct["struct_params"]["model_nig_beta"])           # scalar > 0
    mu0 = jnp.asarray(struct["struct_params"]["model_nig_mu"])            # scalar
    lam0 = jnp.asarray(struct["struct_params"]["model_nig_lam"])          # scalar > 0

    # component params (initialize from model return if present, else random)
    if "model_Reg" in struct["struct_dists"] and len(struct["struct_dists"]["model_Reg"]) == K:
        mu_k = jnp.array([struct["struct_dists"]["model_Reg"][k][0] for k in range(K)])         # (K,)
        sig2_k = jnp.array([struct["struct_dists"]["model_Reg"][k][1] for k in range(K)])       # (K,)
    else:
        rng_key, k1, k2 = random.split(rng_key, 3)
        sig2_k = 1.0 / random.gamma(k1, a0, (K,)) * b0
        mu_k   = mu0 + random.normal(k2, (K,)) * jnp.sqrt(sig2_k / lam0)

    if "model_Gen" in struct["struct_dists"] and len(struct["struct_dists"]["model_Gen"]) == K:
        phi_k = jnp.stack(struct["struct_dists"]["model_Gen"], axis=0)  # (K, V)
    else:
        rng_key, kphi = random.split(rng_key)
        phi_k = random.dirichlet(kphi, beta_dir, (K,))                  # (K, V)

    # ----- initialize latents -------------------------------------------------
    if init_state is not None:
        z_S   = jnp.array(init_state["z_S"])         # (N,)
        z_C   = jnp.array(init_state["z_C"])         # (N,)
        z_Reg = jnp.array(init_state["z_Reg"])       # (N,)
        z_Gen = jnp.array(init_state["z_Gen"])       # (N, M)
        g_i   = jnp.array(init_state["g"])           # (N, K)
    else:
        rng_key, kS, kC, kR, kG = random.split(rng_key, 5)
        z_S   = _sample_categorical(kS, _safe_log(jnp.tile(pi_S[None], (N,1))))
        z_C   = jnp.array([_sample_categorical(random.fold_in(kC, i), _safe_log(pi_C[int(z_S[i])][None, :])) for i in range(N)])
        z_Reg = _sample_categorical(kR, _safe_log(jnp.ones((N, K)) / K))
        z_Gen = jnp.array([
            _sample_categorical(random.fold_in(kG, i), _safe_log(jnp.ones((M, K)) / K))
            for i in range(N)
        ])
        # sample initial g_i from base G2_{s,c}
        gi_list = []
        for i in range(N):
            s, c = int(z_S[i]), int(z_C[i])
            rng_key, ki = random.split(rng_key)
            gi_list.append(random.dirichlet(ki, G2[(s, c)]))
        g_i = jnp.stack(gi_list, axis=0)  # (N, K)

    # ---- storage (optional) --------------------------------------------------
    if not record_last_only:
        keep_idx = []
        trace = {
            "z_S": [], "z_C": [], "z_Reg": [], "z_Gen": [],
            "g": [], "mu_k": [], "sig2_k": [], "phi_k": []
        }

    # ---- Gibbs sweeps --------------------------------------------------------
    for t in range(num_iters):
        # 1) Resample phi_k (words)  ~ Dir(beta + counts)
        wc = _word_counts_for_topic(z_Gen, features, K, V)  # (K, V)
        rng_key, kphi = random.split(rng_key)
        phi_k = random.dirichlet(kphi, beta_dir + wc)

        # 2) Resample (mu_k, sigma2_k) for regression components using NIG conjugacy
        for k in range(K):
            yk = labels[z_Reg == k]
            mu_n, lam_n, a_n, b_n = _posterior_nig(yk, mu0, lam0, a0, b0)
            # sample:
            rng_key, k1, k2 = random.split(rng_key, 3)
            # sigma^2 ~ InvGamma(a_n, b_n)  (sample as 1 / Gamma(a_n, 1/b_n))
            sig2 = b_n / random.gamma(k1, a_n)
            mu   = mu_n + random.normal(k2) * jnp.sqrt(sig2 / lam_n)
            sig2_k = sig2_k.at[k].set(sig2)
            mu_k   = mu_k.at[k].set(mu)

        # 3) For each item i, resample g_i | z_Reg[i], z_Gen[i,*] and (s,c)
        #    posterior Dir( alpha_sc * G2_{s,c} + counts_i ), where counts_i is K-vector from (z_Reg and z_Gen tokens)
        counts_i = jnp.zeros((N, K))
        # add one for regression component
        counts_i = counts_i.at[jnp.arange(N), z_Reg].add(1)
        # add token counts
        for i in range(N):
            ci = jnp.bincount(z_Gen[i], length=K)
            counts_i = counts_i.at[i].add(ci)

        gi_list = []
        for i in range(N):
            s, c = int(z_S[i]), int(z_C[i])
            alpha_sc_base = G2[(s, c)]  # (K,)
            rng_key, ki = random.split(rng_key)
            gi_list.append(random.dirichlet(ki, alpha_sc_base + counts_i[i]))
        g_i = jnp.stack(gi_list, axis=0)  # (N, K)

        # 4) Resample z_Reg[i] ~ Cat( proportional to g_i[k] * Normal(y_i | mu_k, sig2_k) )
        log_like_reg = jnp.stack([_log_normal_pdf(labels, mu_k[k], sig2_k[k]) for k in range(K)], axis=1)  # (N, K)
        logp_reg = _safe_log(g_i) + log_like_reg
        rng_key, kR = random.split(rng_key)
        z_Reg = _sample_categorical(kR, logp_reg)

        # 5) Resample z_Gen[i, m] ~ Cat( proportional to g_i[k] * Multinomial1(x_im | phi_k) )
        #     For one-hot word x_im, the likelihood reduces to phi_k[word_index]
        #     We compute word index with argmax over V (assumed one-hot)
        word_idx = jnp.argmax(features, axis=2)  # (N, M)
        zG_new = []
        for i in range(N):
            im = []
            for m in range(M):
                w = int(word_idx[i, m])
                logp = _safe_log(g_i[i]) + _safe_log(phi_k[:, w])  # (K,)
                rng_key, km = random.split(rng_key)
                im.append(int(_sample_categorical(km, logp[None, :])))
            zG_new.append(jnp.array(im, dtype=jnp.int32))
        z_Gen = jnp.stack(zG_new, axis=0)

        # 6) Resample (z_S[i], z_C[i]) jointly using:
        #     p(s,c | rest) ∝ pi_S[s] * pi_C[s][c] * Dirichlet_pdf( g_i | alpha_sc * G2_{s,c} )
        zS_new, zC_new = [], []
        for i in range(N):
            logp_sc = []
            for s in range(S):
                base_s = _safe_log(pi_S[s])
                for c in range(C):
                    base_sc = _safe_log(pi_C[s][c])
                    # treat alpha_sc as a scalar absorbed into G2 prior already (your model uses alpha*base in sampling);
                    # here we simply use the base vector from model return as prior shape
                    log_dir = _dirichlet_log_pdf(g_i[i], G2[(s, c)])
                    logp_sc.append(base_s + base_sc + log_dir)
            logp_sc = jnp.array(logp_sc)  # length S*C
            rng_key, kc = random.split(rng_key)
            idx = int(_sample_categorical(kc, logp_sc[None, :]))
            s_star, c_star = idx // C, idx % C
            zS_new.append(s_star)
            zC_new.append(c_star)
        z_S = jnp.array(zS_new, dtype=jnp.int32)
        z_C = jnp.array(zC_new, dtype=jnp.int32)

        # 7) (Optional) Resample cluster priors pi_S, pi_C from Dirichlet using counts
        #     Conjugate update: Dir(eta + counts)
        #     If you want them fixed, comment this block out.
        eta_S = jnp.asarray(struct["cluster_params"]["model_eta_G0"])      # (S,)
        eta_C = {s: jnp.asarray(struct["cluster_params"][f"model_eta_G1_S_{s}"]) for s in range(S)}  # (C,)
        # counts for S
        cntS = jnp.bincount(z_S, length=S)
        rng_key, kpS = random.split(rng_key)
        pi_S = random.dirichlet(kpS, eta_S + cntS)
        # counts for C within S
        for s in range(S):
            cntC = jnp.bincount(z_C[z_S == s], length=C)
            rng_key, kpC = random.split(rng_key)
            pi_C[s] = random.dirichlet(kpC, eta_C[s] + cntC)

        # ---- record -----------------------------------------------------------
        if not record_last_only:
            if t >= burnin and ((t - burnin) % thin == 0):
                for name, arr in [("z_S", z_S), ("z_C", z_C), ("z_Reg", z_Reg),
                                  ("z_Gen", z_Gen), ("g", g_i),
                                  ("mu_k", mu_k), ("sig2_k", sig2_k), ("phi_k", phi_k)]:
                    trace[name].append(arr)
                keep_idx.append(t)

    # ---- return --------------------------------------------------------------
    last_state = {
        "z_S": z_S, "z_C": z_C, "z_Reg": z_Reg, "z_Gen": z_Gen, "g": g_i,
        "mu_k": mu_k, "sigma2_k": sig2_k, "phi_k": phi_k,
        # also return updated cluster priors if you chose to resample them
        "pi_S": pi_S, "pi_C": pi_C,
    }
    if record_last_only:
        return last_state
    else:
        # stack trace arrays
        for k in trace:
            trace[k] = jnp.stack(trace[k], axis=0)
        return {"last": last_state, "trace": trace, "iters_kept": jnp.array(keep_idx)}


In [ ]:
import jax
import jax.numpy as jnp
from jax import random
import matplotlib.pyplot as plt

# reuse helpers from previous code (_safe_log, _log_normal_pdf, _dirichlet_log_pdf, etc.)

def _log_likelihood(features, labels, z_S, z_C, z_Reg, z_Gen, g_i, mu_k, sig2_k, phi_k):
    """Compute complete-data log likelihood."""
    N, M, V = features.shape
    K = mu_k.shape[0]

    # regression likelihood
    log_like_reg = 0.
    for i in range(N):
        k = int(z_Reg[i])
        log_like_reg += _log_normal_pdf(labels[i], mu_k[k], sig2_k[k])

    # word likelihood
    word_idx = jnp.argmax(features, axis=2)  # (N, M)
    log_like_words = 0.
    for i in range(N):
        for m in range(M):
            k = int(z_Gen[i, m])
            w = int(word_idx[i, m])
            log_like_words += _safe_log(phi_k[k, w])

    return float(log_like_reg + log_like_words)

def gibbs_hdmm_monitor(
    rng_key,
    model_ret,
    num_iters=200,
    burnin=100,
    thin=1,
):
    """Run Gibbs sampling and record log likelihood + posterior params."""

    # ---- run Gibbs (using previous gibbs_hdmm code) ----
    trace = gibbs_hdmm(
        rng_key,
        model_ret,
        num_iters=num_iters,
        burnin=0,       # record from start
        thin=1,
        record_last_only=False,
    )

    # ---- compute log likelihoods ----
    loglikes = []
    for t in range(trace["trace"]["z_S"].shape[0]):
        z_S   = trace["trace"]["z_S"][t]
        z_C   = trace["trace"]["z_C"][t]
        z_Reg = trace["trace"]["z_Reg"][t]
        z_Gen = trace["trace"]["z_Gen"][t]
        g     = trace["trace"]["g"][t]
        mu_k  = trace["trace"]["mu_k"][t]
        sig2_k= trace["trace"]["sig2_k"][t]
        phi_k = trace["trace"]["phi_k"][t]
        features = model_ret["observed_variables"]["features"]
        labels   = model_ret["observed_variables"]["labels"]

        ll = _log_likelihood(features, labels, z_S, z_C, z_Reg, z_Gen, g, mu_k, sig2_k, phi_k)
        loglikes.append(ll)

    loglikes = jnp.array(loglikes)

    # ---- plots ----
    fig, axs = plt.subplots(2, 2, figsize=(12, 10))

    # log likelihood trace
    axs[0, 0].plot(loglikes, lw=1)
    axs[0, 0].set_title("Log Likelihood Trace")
    axs[0, 0].set_xlabel("Iteration")
    axs[0, 0].set_ylabel("Log Likelihood")

    # posterior mu_k trace
    axs[0, 1].plot(trace["trace"]["mu_k"])
    axs[0, 1].set_title("Posterior Means (mu_k)")
    axs[0, 1].set_xlabel("Iteration")

    # posterior sigma² trace
    axs[1, 0].plot(trace["trace"]["sig2_k"])
    axs[1, 0].set_title("Posterior Variances (sigma²_k)")
    axs[1, 0].set_xlabel("Iteration")

    # heatmap of word-topic distributions φ_k
    final_phi = trace["trace"]["phi_k"][-1]  # last sample
    im = axs[1, 1].imshow(final_phi, aspect="auto", cmap="viridis")
    axs[1, 1].set_title("Topic-Word Distribution (last sample)")
    axs[1, 1].set_xlabel("Word index")
    axs[1, 1].set_ylabel("Topic index")
    fig.colorbar(im, ax=axs[1, 1])

    plt.tight_layout()
    plt.show()

    return {
        "trace": trace,
        "loglikes": loglikes,
    }


In [ ]:

model_return = hdmm_model(data, struct_upbd, vocab_size)
key = jax.random.PRNGKey(0)
out = gibbs_hdmm_monitor(key, model_return, num_iters=500, burnin=100)

# access traces
trace = out["trace"]
loglikes = out["loglikes"]


In [ ]:
# Gibbs sampler for hdmm_model (fixed & documented)
# Requires: jax, jax.numpy, matplotlib
import jax
import jax.numpy as jnp
from jax import random
from jax.scipy.special import gammaln
import matplotlib.pyplot as plt

# ------------------ helpers ------------------

def _safe_log(x, eps=1e-12):
    return jnp.log(jnp.clip(x, eps, None))

def _log_normal_pdf(y, mu, var):
    # scalar y or vector; returns log p(y | mu, var)
    return -0.5 * (_safe_log(2*jnp.pi*var) + (y - mu) ** 2 / var)

def _dirichlet_log_pdf(x, alpha):
    # log p(x | alpha) for Dirichlet(alpha), x on simplex
    # log Dir(x; alpha) = ln Gamma(sum alpha) - sum ln Gamma(alpha_i) + sum (alpha_i - 1) ln x_i
    alpha0 = jnp.sum(alpha)
    return (gammaln(alpha0) - jnp.sum(gammaln(alpha))) + jnp.sum((alpha - 1.0) * _safe_log(x))

def _sample_categorical(key, log_probs):
    # log_probs shape (..., K) or (K,) ; returns integer(s)
    # random.categorical expects unnormalized log-probs
    return random.categorical(key, log_probs, axis=-1)

def _one_hot(idx, K):
    return jnp.eye(K, dtype=jnp.int32)[idx]

def _word_counts_for_topic(z_gen, features, K, V):
    # naive loop version: z_gen (N, M), features (N, M, V) one-hot
    counts = jnp.zeros((K, V))
    for k in range(K):
        mask = (z_gen == k)[..., None]  # (N, M, 1)
        counts = counts.at[k].set(jnp.sum(features * mask, axis=(0,1)))
    return counts  # (K, V)

def _posterior_nig(y_k, mu0, lam0, a0, b0):
    # Returns posterior NIG params (mu_n, lam_n, a_n, b_n)
    n = y_k.shape[0]
    if n == 0:
        return mu0, lam0, a0, b0
    ybar = jnp.mean(y_k)
    sse = jnp.sum((y_k - ybar)**2)
    lam_n = lam0 + n
    mu_n  = (lam0*mu0 + n*ybar) / lam_n
    a_n   = a0 + 0.5*n
    b_n   = b0 + 0.5*sse + 0.5*(lam0*n/lam_n) * (ybar - mu0)**2
    return mu_n, lam_n, a_n, b_n

# ------------------ Gibbs sampler (fixed) ------------------

def gibbs_hdmm_fixed_monitor(
    rng_key,
    model_ret,
    num_iters=500,
    burnin=100,
    thin=1,
    record_every=1,
):
    """
    Gibbs sampler with corrected Dirichlet prior shapes and monitoring.
    Inputs:
      - rng_key: jax.random.PRNGKey
      - model_ret: dict returned by hdmm_model (contains struct_params, struct_dists, cluster_dists, observed_variables)
      - num_iters, burnin, thin: sampling control
      - record_every: integer, store every record_every-th sample (controls trace size)
    Returns:
      - dict with traces, final state, and plotted figures (plots are shown inline).
    """

    # ----- unpack model -----
    struct = model_ret
    upbd = struct["struct_assumptions"]
    K = upbd["G0"]
    S = upbd["G1"]
    C = upbd["G2"]

    features = struct["observed_variables"]["features"]  # (N, M, V) assumed one-hot
    labels   = struct["observed_variables"]["labels"]    # (N,)
    N, M, V  = features.shape

    # cluster priors (initial)
    pi_S = jnp.asarray(struct["cluster_dists"]["model_G0"])               # (S,)
    pi_C = {s: jnp.asarray(struct["cluster_dists"][f"model_G1_S_{s}"])    # (C,)
           for s in range(S)}

    # G2 base distributions (these are vectors on the simplex)
    G2 = {(s, c): jnp.asarray(struct["struct_dists"][f"model_G2_S_{s}_C_{c}"])
          for s in range(S) for c in range(C)}

    # alpha vectors for combining with base G2; should exist in struct_params
    # key name used in original hdmm_model: model_alpha_S_{s}_C_{c}_G
    alpha_sc = {}
    for s in range(S):
        for c in range(C):
            key = f"model_alpha_S_{s}_C_{c}_G"
            if key in struct["struct_params"]:
                alpha_sc[(s, c)] = jnp.asarray(struct["struct_params"][key])
            else:
                # fallback to vector of ones(K)
                alpha_sc[(s, c)] = jnp.ones(K)

    # phi priors and NIG priors for regression
    beta_dir = jnp.asarray(struct["struct_params"]["model_beta"])         # (V,)
    a0 = jnp.asarray(struct["struct_params"]["model_nig_alpha"])
    b0 = jnp.asarray(struct["struct_params"]["model_nig_beta"])
    mu0 = jnp.asarray(struct["struct_params"]["model_nig_mu"])
    lam0 = jnp.asarray(struct["struct_params"]["model_nig_lam"])

    # initialize or take existing component params
    if "model_Reg" in struct["struct_dists"] and len(struct["struct_dists"]["model_Reg"]) == K:
        mu_k = jnp.array([struct["struct_dists"]["model_Reg"][k][0] for k in range(K)])         # (K,)
        sig2_k = jnp.array([struct["struct_dists"]["model_Reg"][k][1] for k in range(K)])       # (K,)
    else:
        rng_key, k1, k2 = random.split(rng_key, 3)
        sig2_k = 1.0 / random.gamma(k1, a0, (K,)) * b0
        mu_k   = mu0 + random.normal(k2, (K,)) * jnp.sqrt(sig2_k / lam0)

    if "model_Gen" in struct["struct_dists"] and len(struct["struct_dists"]["model_Gen"]) == K:
        phi_k = jnp.stack(struct["struct_dists"]["model_Gen"], axis=0)  # (K, V)
    else:
        rng_key, kphi = random.split(rng_key)
        phi_k = random.dirichlet(kphi, beta_dir, (K,))                  # (K, V)

    # ----- initialize latents -----
    rng_key, kS, kC, kR, kG = random.split(rng_key, 5)
    z_S = _sample_categorical(kS, _safe_log(jnp.tile(pi_S[None], (N,1))))   # (N,)
    z_C = jnp.array([_sample_categorical(random.fold_in(kC, i), _safe_log(pi_C[int(z_S[i])][None,:])) for i in range(N)])
    # init z_Reg and z_Gen uniformly
    rng_key, kR2, kG2 = random.split(rng_key, 3)
    z_Reg = _sample_categorical(kR2, _safe_log(jnp.ones((N, K))/K))
    # z_Gen: (N, M) categorical per token; sample per-item
    def sample_zG_for_i(key, M, K):
        # returns (M,) ints
        return jnp.array([int(random.categorical(random.fold_in(key, m), jnp.zeros(K))) for m in range(M)])
    z_Gen = jnp.stack([sample_zG_for_i(random.fold_in(kG2, i), M, K) for i in range(N)], axis=0)

    # initial g_i from prior alpha_sc * G2_{s,c}
    g_list = []
    for i in range(N):
        s, c = int(z_S[i]), int(z_C[i])
        alpha_vec = alpha_sc[(s, c)]                       # (K,)
        base = G2[(s, c)]                                  # (K,) simplex
        prior_shape = alpha_vec * base                     # <-- CORRECT prior shape (elementwise)
        rng_key, kg = random.split(rng_key)
        g_list.append(random.dirichlet(kg, prior_shape))
    g_i = jnp.stack(g_list, axis=0)  # (N, K)

    # ----- traces & diagnostics -----
    records = []
    loglikes = []
    mu_trace = []
    sig2_trace = []
    phi_trace = []

    # compute word indices (assume one-hot)
    word_idx = jnp.argmax(features, axis=2)  # (N, M)

    # iterate
    for t in range(num_iters):
        # ---------- (A) Update phi_k (topic-word multinomials)
        # Posterior: phi_k ~ Dir( beta + n_{k,1:V} )
        # where n_{k,v} = sum_{i,m} 1{z_Gen[i,m]=k and x_{i,m}=v}
        wc = _word_counts_for_topic(z_Gen, features, K, V)  # (K, V)
        rng_key, kphi = random.split(rng_key)
        phi_k = random.dirichlet(kphi, beta_dir + wc)

        # ---------- (B) Update regression component params (Normal-Inverse-Gamma conjugacy)
        # For component k with data y_{k,1:n_k}:
        # lam_n = lam0 + n_k
        # mu_n = (lam0*mu0 + n_k * ybar) / lam_n
        # a_n = a0 + n_k/2
        # b_n = b0 + 1/2 sum (y - ybar)^2 + (lam0 * n_k / (2 lam_n)) (ybar - mu0)^2
        # then sigma2 ~ InvGamma(a_n, b_n), mu ~ Normal(mu_n, sigma2/lam_n)
        for k in range(K):
            yk = labels[z_Reg == k]
            mu_n, lam_n, a_n, b_n = _posterior_nig(yk, mu0, lam0, a0, b0)
            rng_key, r1, r2 = random.split(rng_key, 3)
            # InvGamma(a_n, b_n): sample as b_n / Gamma(a_n)
            # (Note: jax.random.gamma(k, shape) returns Gamma(shape) with scale=1.)
            sig2 = b_n / random.gamma(r1, a_n) if a_n > 0 else b_n
            mu   = mu_n + random.normal(r2) * jnp.sqrt(sig2 / lam_n)
            sig2_k = sig2_k.at[k].set(sig2)
            mu_k   = mu_k.at[k].set(mu)

        # ---------- (C) Update per-item mixture g_i
        # Prior: g_i ~ Dir( alpha_sc * G2_{s,c} )
        # Likelihood contribution from assigned latents: counts_i[k] = 1{z_Reg[i]=k} + sum_m 1{z_Gen[i,m]=k}
        # Posterior: g_i ~ Dir( alpha_sc * G2 + counts_i )
        counts_i = jnp.zeros((N, K))
        counts_i = counts_i.at[jnp.arange(N), z_Reg].add(1)  # regression assignment counts
        for i in range(N):
            ci = jnp.bincount(z_Gen[i], length=K)
            counts_i = counts_i.at[i].add(ci)

        g_new_list = []
        for i in range(N):
            s, c = int(z_S[i]), int(z_C[i])
            alpha_vec = alpha_sc[(s, c)]
            base = G2[(s, c)]
            prior_shape = alpha_vec * base                     # PRIOR SHAPE used in original model
            post_shape = prior_shape + counts_i[i]            # posterior Dirichlet parameters
            rng_key, kg = random.split(rng_key)
            g_new_list.append(random.dirichlet(kg, post_shape))
        g_i = jnp.stack(g_new_list, axis=0)

        # ---------- (D) Update z_Reg (regression assignment)
        # p(z_Reg[i]=k | ...) ∝ g_i[i,k] * N(y_i | mu_k, sigma2_k)
        log_like_reg = jnp.stack([_log_normal_pdf(labels, mu_k[k], sig2_k[k]) for k in range(K)], axis=1)  # (N, K)
        logp_reg = _safe_log(g_i) + log_like_reg
        rng_key, kr = random.split(rng_key)
        z_Reg = _sample_categorical(kr, logp_reg)

        # ---------- (E) Update z_Gen (token-topic assignments)
        # p(z_{i,m}=k | ...) ∝ g_i[k] * phi_k[w]
        zG_new = []
        for i in range(N):
            im = []
            for m in range(M):
                w = int(word_idx[i, m])
                logp = _safe_log(g_i[i]) + _safe_log(phi_k[:, w])  # (K,)
                rng_key, ktmp = random.split(rng_key)
                im.append(int(_sample_categorical(ktmp, logp[None, :])))
            zG_new.append(jnp.array(im, dtype=jnp.int32))
        z_Gen = jnp.stack(zG_new, axis=0)

        # ---------- (F) Update hierarchy labels (z_S, z_C) jointly
        # p(s,c | g_i, pi_S, pi_C, alpha_sc, G2) ∝ pi_S[s] * pi_C[s][c] * DirPDF(g_i | alpha_sc * G2_{s,c})
        zS_new = []
        zC_new = []
        for i in range(N):
            logp_sc = []
            for s in range(S):
                for c in range(C):
                    log_prior_sc = _safe_log(pi_S[s]) + _safe_log(pi_C[s][c])
                    prior_shape = alpha_sc[(s, c)] * G2[(s, c)]
                    log_dir = _dirichlet_log_pdf(g_i[i], prior_shape)
                    logp_sc.append(log_prior_sc + log_dir)
            logp_sc = jnp.array(logp_sc)
            rng_key, ksc = random.split(rng_key)
            idx = int(_sample_categorical(ksc, logp_sc[None, :]))
            s_star, c_star = idx // C, idx % C
            zS_new.append(s_star)
            zC_new.append(c_star)
        z_S = jnp.array(zS_new, dtype=jnp.int32)
        z_C = jnp.array(zC_new, dtype=jnp.int32)

        # ---------- (G) Optionally update cluster priors pi_S, pi_C (Dirichlet conjugacy)
        eta_S = jnp.asarray(struct["cluster_params"]["model_eta_G0"])
        eta_C = {s: jnp.asarray(struct["cluster_params"][f"model_eta_G1_S_{s}"]) for s in range(S)}
        cntS = jnp.bincount(z_S, length=S)
        rng_key, pSkey = random.split(rng_key)
        pi_S = random.dirichlet(pSkey, eta_S + cntS)
        for s in range(S):
            cntC = jnp.bincount(z_C[z_S == s], length=C)
            rng_key, pCkey = random.split(rng_key)
            pi_C[s] = random.dirichlet(pCkey, eta_C[s] + cntC)

        # ---------- diagnostics & recording ----------
        if (t >= burnin) and ((t - burnin) % thin == 0):
            # compute log joint-like quantity (only lik terms for now)
            # Log-likelihood = sum_i log p(y_i | z_Reg_i, mu, sigma2) + sum_{i,m} log phi_{z_{i,m}}(w)
            ll_reg = jnp.sum(jnp.stack([_log_normal_pdf(labels, mu_k[k], sig2_k[k]) for k in range(K)], axis=1)[jnp.arange(N), z_Reg])
            ll_words = 0.0
            for i in range(N):
                for m in range(M):
                    k = int(z_Gen[i, m]); w = int(word_idx[i,m])
                    ll_words += _safe_log(phi_k[k, w])
            loglikes.append(float(ll_reg + ll_words))

            mu_trace.append(jnp.array(mu_k))
            sig2_trace.append(jnp.array(sig2_k))
            phi_trace.append(jnp.array(phi_k))

        # continue loop

    # stack traces
    loglikes = jnp.array(loglikes)
    mu_trace = jnp.stack(mu_trace, axis=0)       # (T, K)
    sig2_trace = jnp.stack(sig2_trace, axis=0)   # (T, K)
    phi_trace = jnp.stack(phi_trace, axis=0)     # (T, K, V)

    final_state = {
        "z_S": z_S, "z_C": z_C, "z_Reg": z_Reg, "z_Gen": z_Gen, "g": g_i,
        "mu_k": mu_k, "sigma2_k": sig2_k, "phi_k": phi_k,
        "pi_S": pi_S, "pi_C": pi_C,
    }

    # ------------------ Plots ------------------
    fig, axs = plt.subplots(3, 1, figsize=(10, 12))
    axs[0].plot(loglikes)
    axs[0].set_title("Log Likelihood (likelihood of labels + words) after burnin")
    axs[0].set_xlabel("Saved iteration")
    axs[0].set_ylabel("Log Likelihood")

    # plot mu traces for first few topics (or all if small K)
    axs[1].plot(mu_trace)
    axs[1].set_title("Trace of mu_k (posterior means)")
    axs[1].set_xlabel("Saved iteration")

    axs[2].plot(sig2_trace)
    axs[2].set_title("Trace of sigma^2_k (posterior variances)")
    axs[2].set_xlabel("Saved iteration")
    plt.tight_layout()
    plt.show()

    # heatmap of final phi_k
    fig2, ax2 = plt.subplots(1, 1, figsize=(10, 6))
    ax2.imshow(phi_trace[-1], aspect='auto')
    ax2.set_title("Topic-word distribution phi_k (last saved sample)")
    ax2.set_xlabel("Word index")
    ax2.set_ylabel("Topic index")
    plt.show()

    # running posterior means (smoothed)
    running_mu = jnp.cumsum(mu_trace, axis=0) / (jnp.arange(1, mu_trace.shape[0]+1)[:, None])
    fig3, ax3 = plt.subplots(1, 1, figsize=(8,5))
    ax3.plot(running_mu)
    ax3.set_title("Running posterior mean of mu_k")
    plt.show()

    return {
        "final_state": final_state,
        "loglikes": loglikes,
        "mu_trace": mu_trace,
        "sig2_trace": sig2_trace,
        "phi_trace": phi_trace,
    }
